<a href="https://colab.research.google.com/github/kdkim2000/12week-health-tracker/blob/main/%5B%EA%B0%95%EC%9D%98%EB%82%B4%EC%9A%A9%5D_3_LangChain%EC%9D%84_%EC%9D%B4%EC%9A%A9%ED%95%9C_%EB%8D%B0%EC%9D%B4%ED%84%B0_%EC%83%9D%EC%84%B1%EA%B3%BC_%EC%B2%98%EB%A6%AC_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [실습] LangChain을 이용한 데이터 생성과 처리


LCEL의 기본 문법인 Prompt | llm | Parser 구조에 대해 배웠습니다.   
이번 실습에서는 출력을 구조화하고, LLM을 연결하는 방법에 대해 알아봅니다.


### 라이브러리 설치  

랭체인 OpenAI 모듈을 설치합니다.

In [1]:
%pip install langchain langchain_openai dotenv rich -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 17.4 MB/s eta 0:00:00


In [2]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

if os.environ.get('OPENAI_API_KEY'):
    print('OpenAI API 키 확인')

OpenAI API 키 확인


### LLM 모델 불러오기

In [3]:
from langchain.chat_models import init_chat_model

gpt_llm = init_chat_model(
    "gpt-5.2", reasoning_effort='low')

## JsonOutputParser 로 Json 형식의 출력 만들기

LLM의 출력을 구조화하면, 데이터 후처리를 하지 않고도 다른 코드와 연결할 수 있습니다.   
JSON 형식의 출력을 구성해 보겠습니다.

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import JsonOutputParser

jsonparser = JsonOutputParser()

JSON 파서의 역할은 JSON 규격에 맞는 텍스트를 Dict 형식으로 변환하는 것으로,     
실제 형식에 대한 조건을 프롬프트로 전달해야 합니다.

In [5]:
jsonparser.get_format_instructions()

'Return a JSON object.'

In [6]:
recipe_template = ChatPromptTemplate([
    ('system','당신은 전세계의 이색적인 퓨전 조리법의 전문가입니다.'),
    ('user','''저는 다음의 재료와 조건을 이용한 환상적인
퓨전 다이닝을 만들고 싶습니다. 1가지 메뉴만 추천해주세요!
레시피에 대한 정보를 JSON 형식으로 출력해주세요.

[재료]: {ingredient}
[조건]: {condition}
''')
])

recipe_chain = recipe_template | gpt_llm | jsonparser

In [7]:
response = recipe_chain.invoke({'ingredient':'콜라, 고수, 새우', 'condition':'디저트'})
response

{'menu_name': '콜라 카라멜 새우 브리틀 & 고수-라임 그라니타',
 'type': 'dessert',
 'fusion_concept': '콜라를 졸여 만든 스파이스 카라멜에 새우를 캔디드(단짠)하게 코팅해 브리틀로 굳히고, 고수와 라임의 향을 살린 그라니타로 상큼함을 더한 이색 디저트.',
 'servings': 2,
 'time': {'prep_minutes': 15, 'cook_minutes': 20, 'freeze_minutes': 120},
 'ingredients': [{'name': '콜라',
   'amount': '330ml',
   'notes': '제로/다이어트보단 일반 콜라 권장(카라멜라이징에 유리)'},
  {'name': '새우(중하, 껍질 제거·내장 제거)', 'amount': '120g', 'notes': '키친타월로 물기 완전 제거'},
  {'name': '고수(잎 위주)', 'amount': '15g', 'notes': '일부는 장식용으로 따로 보관'},
  {'name': '설탕', 'amount': '25g', 'notes': '그라니타용'},
  {'name': '라임', 'amount': '1개', 'notes': '즙 20~25ml + 제스트 약간'},
  {'name': '소금', 'amount': '한 꼬집', 'notes': '단맛을 올리고 새우 비린내를 잡음'},
  {'name': '버터', 'amount': '10g', 'notes': '콜라 카라멜에 윤기 부여(선택)'},
  {'name': '중성유', 'amount': '약간', 'notes': '새우를 굽기 위한 최소량'}],
 'steps': [{'step': 1,
   'title': '고수-라임 그라니타 베이스 만들기',
   'details': '블렌더에 고수 12g, 라임즙, 설탕 25g, 물 200ml, 소금 한 꼬집을 넣고 곱게 간다. 체에 한 번 걸러 얇은 트레이에 붓고 냉동한다.'},
  {'step': 2,
   'titl

In [8]:
# Dict 구조: 추출 가능
response['menu_name']

'콜라 카라멜 새우 브리틀 & 고수-라임 그라니타'

Json으로 파싱하는 방법은 활용도가 높지만, 실행할 때마다 결과뿐만 아니라 형식도 달라진다는 문제가 있습니다.

In [9]:
response = recipe_chain.invoke({'ingredient':'문어, 피넛버터', 'condition':'메인 요리'})
response

{'menu_name': '피넛 사테 글레이즈 문어 스테이크 & 코코넛 라임 쿠스쿠스',
 'type': '메인 요리',
 'fusion_concept': '지중해식 문어 스테이크에 동남아 사테(피넛버터) 글레이즈를 입혀 숯불(또는 팬) 향을 더하고, 코코넛-라임 쿠스쿠스로 상큼하게 균형을 맞춘 퓨전 다이닝 플레이트',
 'servings': 2,
 'ingredients': {'octopus': [{'item': '문어(손질된 것, 삶기용)', 'amount': '700 g'},
   {'item': '월계수잎', 'amount': '1장'},
   {'item': '통후추', 'amount': '1/2 tsp'},
   {'item': '레몬(삶는 물 향)', 'amount': '1/2개'}],
  'peanut_satay_glaze': [{'item': '피넛버터(무가당 또는 크리미)',
    'amount': '70 g (약 4 tbsp)'},
   {'item': '간장', 'amount': '2 tbsp'},
   {'item': '라임즙(또는 레몬즙)', 'amount': '1.5 tbsp'},
   {'item': '꿀 또는 흑설탕', 'amount': '1 tbsp'},
   {'item': '다진 마늘', 'amount': '1 tsp'},
   {'item': '생강 간 것', 'amount': '1 tsp'},
   {'item': '고추가루 또는 스리라차', 'amount': '1 tsp (기호)'},
   {'item': '코코넛밀크 또는 물', 'amount': '3–5 tbsp (농도 조절)'},
   {'item': '참기름', 'amount': '1 tsp'}],
  'coconut_lime_couscous': [{'item': '쿠스쿠스', 'amount': '150 g'},
   {'item': '코코넛밀크', 'amount': '200 ml'},
   {'item': '물', 'amount': '100 ml'},
   {

## Pydantic을 이용해 출력 형식 지정하기

pydantic은 데이터 형식에 제약조건을 두고 이를 준수하는지 검증하는 라이브러리입니다.


In [10]:
from pydantic import BaseModel, Field
# pydantic 연동

class Recipe(BaseModel):
    name: str = Field(description="음식 이름")
    # name: 문자열, 설명은 "음식 이름"
    difficulty: str = Field(description="만들기의 난이도")

    origin: str = Field(description="원산지")
    ingredients: list[str] = Field(description="재료")
    # ingredients: 문자열 리스트, 설명은 "재료"

    instructions: list[str] = Field(description="조리법")
    tip: str = Field(description='실패하는 5가지 시나리오')


In [11]:
parser = JsonOutputParser(pydantic_object=Recipe)

In [12]:
print(parser.get_format_instructions())

STRICT OUTPUT FORMAT:
- Return only the JSON value that conforms to the schema. Do not include any additional text, explanations, headings, or separators.
- Do not wrap the JSON in Markdown or code fences (no ``` or ```json).
- Do not prepend or append any text (e.g., do not write "Here is the JSON:").
- The response must be a single top-level JSON value exactly as required by the schema (object/array/etc.), with no trailing commas or comments.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]} the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema (shown in a code block for readability only — do not include any backticks or Markdown in your output):


해당 내용을 프롬프트에 포함합니다.

In [13]:
recipe_template2 = ChatPromptTemplate([
    ('system','당신은 전세계의 이색적인 퓨전 조리법의 전문가입니다.'),
    ('user','''저는 다음의 재료를 이용한 실험적인 음식을 만들고 싶습니다. 추천해주세요!
    레시피에 대한 정보를 JSON 형식으로 출력해주세요. 결과는 한국어로 작성하세요.
재료: {ingredient}
출력 형식 조건: {instruction}''')
])

recipe_chain2 = recipe_template2 | gpt_llm | parser


In [14]:
recipe_chain2.invoke({'ingredient':'생강', 'instruction':parser.get_format_instructions()})

{'name': '생강-된장 카라멜 글레이즈를 곁들인 구운 가지 스테이크',
 'difficulty': '중급',
 'origin': '한식(된장) + 프렌치(카라멜/글레이즈) + 일본식 감칠맛(미소 계열) 퓨전',
 'ingredients': ['생강(강판에 간 것) 2큰술',
  '된장 1.5큰술',
  '설탕 3큰술',
  '물 2큰술',
  '간장 1큰술',
  '식초 또는 레몬즙 1큰술',
  '버터 1큰술(또는 올리브오일 2큰술)',
  '가지 2개',
  '소금 약간',
  '후추 약간',
  '참깨 또는 견과류(토핑용) 1큰술',
  '쪽파 또는 고수(선택) 약간'],
 'instructions': ['가지는 길게 반으로 자르고, 겉면에 칼집을 격자 모양으로 얕게 넣는다. 소금을 아주 약하게 뿌려 10분 두고, 나온 물기를 닦는다.',
  '팬에 설탕과 물을 넣고 중불에서 저어 녹인 뒤, 젓지 말고 색이 호박색(카라멜색)으로 변할 때까지 끓인다(팬을 살짝 흔들어 색을 맞춘다).',
  '불을 약하게 줄이고 버터(또는 올리브오일)를 넣어 섞은 뒤, 된장과 간 생강을 넣는다. 덩어리가 생기면 약불에서 천천히 풀어준다.',
  "간장과 식초(또는 레몬즙)를 넣어 점도가 '묽은 시럽~소스' 정도가 되게 30초~1분 더 끓인다. 너무 되면 물 1큰술씩 추가한다.",
  '다른 팬(또는 같은 팬을 닦아)에서 가지를 단면부터 중불로 굽는다. 단면이 노릇해지면 뒤집어 껍질 쪽도 굽고, 뚜껑을 덮어 3~5분 부드럽게 익힌다.',
  '불을 약하게 줄이고 가지 위에 생강-된장 카라멜 글레이즈를 넉넉히 바른 뒤 30초~1분 더 구워 표면을 코팅한다(타지 않게 계속 움직인다).',
  '접시에 담고 남은 글레이즈를 한 번 더 끼얹는다. 후추, 참깨(또는 잘게 부순 견과류), 쪽파/고수를 올려 마무리한다.'],
 'tip': '실패하는 5가지 시나리오: (1) 카라멜을 젓다가 설탕이 다시 결정화되어 거칠고 모래알 같은 소스가 됨 → 녹인 뒤에는 젓지 말고 팬만 흔들기.

partial을 통해 먼저 일부를 입력할 수도 있습니다.

In [15]:
recipe_template2 = ChatPromptTemplate([
    ('system','당신은 전세계의 이색적인 퓨전 조리법의 전문가입니다.'),
    ('user','''저는 {ingredient}를 이용한 실험적인 음식을 만들고 싶습니다. 추천해주세요!
    레시피에 대한 정보를 JSON 형식으로 출력해주세요. 결과는 한국어로 작성하세요.
     {instruction}''')
]).partial(instruction=parser.get_format_instructions())

recipe_chain2 = recipe_template2 | gpt_llm | parser

recipe_chain2.invoke('감자')
# partial은 기본값: instruction을 다시 덮어쓸 수도 있음

{'name': '미소-고추장 카라멜 감자 모찌 타코 (김치 사워크림)',
 'difficulty': '중상',
 'origin': '퓨전(일본 모찌+멕시코 타코+한국 장/김치)',
 'ingredients': ['감자 400g(전분 많은 품종 권장)',
  '감자전분 120g',
  '소금 1/2작은술',
  '올리브오일 또는 중성유 1큰술',
  '설탕 3큰술',
  '물 1큰술',
  '버터 20g',
  '된장(미소) 1큰술',
  '고추장 1작은술(매운맛 취향에 따라 가감)',
  '간장 1작은술',
  '식초 1작은술(또는 라임즙)',
  '김치 80g',
  '사워크림 120g(또는 그릭요거트)',
  '꿀 1작은술',
  '후추 약간',
  '쪽파 또는 고수 약간',
  '볶은 김가루 또는 김 시트 약간(선택)',
  '참깨 약간(선택)'],
 'instructions': ['감자를 껍질 벗겨 큼직하게 썰고 소금 약간 넣은 물에 부드럽게 익을 때까지 삶는다. 물기 완전히 빼고 약불에서 1~2분 더 굴려 수분을 날린다.',
  '뜨거울 때 감자를 곱게 으깨고(덩어리 금지) 식힘망이나 넓은 접시에 펼쳐 김을 5분 정도 뺀다.',
  '감자에 감자전분을 2~3번 나눠 넣어 치대며 탄력 있는 반죽을 만든다. 손에 거의 안 붙는 정도가 목표이며, 너무 질면 전분을 1큰술씩 추가한다.',
  '반죽을 8등분해 공 모양으로 만든 뒤, 각각을 지름 10~12cm의 얇은 원형으로 눌러 ‘타코 셸’처럼 접을 수 있게 준비한다.',
  '달군 팬에 기름을 두르고 반죽 원형을 한 장씩 올려 중약불에서 앞뒤로 노릇하게 굽는다. 바로 반으로 접어 타코 모양을 잡고(집게로 30초 고정) 꺼낸다. 남은 것도 반복한다.',
  '카라멜 소스: 냄비에 설탕과 물을 넣고 젓지 말고 중불로 끓여 호박색이 되면 불을 끈다. 버터를 넣어 녹인 뒤 된장, 고추장, 간장, 식초(또는 라임즙)를 넣고 재가열하며 부드럽게 섞어 농도를 맞춘다.',
  '김치 사워크림: 김치를 잘게 다져 물기(

# LangChain Structured Output
파서를 사용하지 않고, 구조화된 출력을 생성합니다.  

In [16]:
from rich import print as rprint
structured_llm = gpt_llm.with_structured_output(Recipe)

rprint(structured_llm)

RunnableSequence(
    first=_ChatModelBinding(
        bound=ChatOpenAI(
            metadata={
                'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.13', 'langchain-openai': '1.4.1'}
            },
            output_version=None,
            profile={
                'name': 'GPT-5.2',
                'release_date': '2025-12-11',
                'last_updated': '2025-12-11',
                'open_weights': False,
                'max_input_tokens': 272000,
                'max_output_tokens': 128000,
                'text_inputs': True,
                'image_inputs': True,
                'audio_inputs': False,
                'video_inputs': False,
                'text_outputs': True,
                'image_outputs': False,
                'audio_outputs': False,
                'video_outputs': False,
                'reasoning_output': True,
                'tool_calling': True,
                'structured_output': True,
                'attachment': True,
                'temperature': False,
                'image_url_inputs': True,
                'pdf_inputs': True,
                'pdf_tool_message': True,
                'image_tool_message': True,
                'tool_choice': True,
                'tool_call_streaming': True,
                'reasoning_effort_levels': ['none', 'low', 'medium', 'high', 'xhigh']
            },
            client=<openai.resources.chat.completions.completions.Completions object at 0x7bedeebb4ef0>,
            async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7bedef025610>,
            root_client=<openai.OpenAI object at 0x7bee2020ee40>,
            root_async_client=<openai.AsyncOpenAI object at 0x7bedeed96f60>,
            model_name='gpt-5.2',
            model_kwargs={},
            openai_api_key=SecretStr('**********'),
            openai_proxy=None,
            stream_usage=True,
            reasoning_effort='low',
            stream_chunk_timeout=120.0
        ),
        kwargs={
            'response_format': <class '__main__.Recipe'>,
            'ls_structured_output_format': {
                'kwargs': {'method': 'json_schema', 'strict': None},
                'schema': {
                    'type': 'function',
                    'function': {
                        'name': 'Recipe',
                        'description': '',
                        'parameters': {
                            'properties': {
                                'name': {'description': '음식 이름', 'type': 'string'},
                                'difficulty': {'description': '만들기의 난이도', 'type': 'string'},
                                'origin': {'description': '원산지', 'type': 'string'},
                                'ingredients': {
                                    'description': '재료',
                                    'items': {'type': 'string'},
                                    'type': 'array'
                                },
                                'instructions': {
                                    'description': '조리법',
                                    'items': {'type': 'string'},
                                    'type': 'array'
                                },
                                'tip': {'description': '실패하는 5가지 시나리오', 'type': 'string'}
                            },
                            'required': ['name', 'difficulty', 'origin', 'ingredients', 'instructions', 'tip'],
                            'type': 'object'
                        }
                    }
                }
            }
        },
        config={},
        config_factories=[]
    ),
    middle=[],
    last=RunnableBinding(
        bound=RunnableLambda(...),
        kwargs={},
        config={},
        config_factories=[],
        custom_output_type=<class '__main__.Recipe'>
    )
)

In [17]:
recipe_template3 = ChatPromptTemplate([
    ('system','당신은 한국 전통의 재료가 가진 다양한 맛과 특성을 활용합니다.'),
    ('user','''저는 {ingredient}를 이용한 실험적인 음식을 만들고 싶습니다. 추천해주세요!''')
])

recipe_chain3 = recipe_template3 | structured_llm
response = recipe_chain3.invoke("생강")
rprint(response)

Recipe(
    name='생강-된장 캐러멜 소스를 곁들인 구운 단호박 & 들기름 요거트',
    difficulty='중',
    origin='한국 전통 재료(생강·된장·조청·들기름)를 현대적으로 재해석',
    ingredients=[
        '단호박 1/2통(또는 고구마/당근으로 대체 가능)',
        '생강 30g(껍질 벗겨 강판에 갈기)',
        '된장 1큰술(구수한 타입 권장)',
        '조청 2큰술(또는 꿀)',
        '간장 1작은술',
        '식초 1작은술(현미식초나 매실식초 추천)',
        '버터 15g(또는 참기름 1작은술로 비건 변형)',
        '물 3~5큰술',
        '플레인 요거트 4큰술(또는 두유 요거트)',
        '들기름 1큰술',
        '소금 한 꼬집',
        '볶은 검은깨 또는 잣 약간',
        '선택: 김가루 약간, 레몬 제스트 약간, 청양고추 아주 소량(매운 포인트)'
    ],
    instructions=[
        '오븐을 200℃로 예열합니다. 단호박은 씨를 긁어내고 2~3cm 두께로 썰어 소금 한 꼬집과 오일(선택)을 아주 살짝 
바른 뒤 25~35분 굽습니다(겉은 카라멜라이즈, 속은 촉촉하게).',
        '소스 팬에 버터를 녹인 뒤 약불에서 간 생강을 30~60초 볶아 향을 올립니다(타지 않게).',
        '된장을 넣고 20~30초 더 볶아 구수한 향을 내고, 조청을 넣어 잘 섞습니다.',
        '간장, 물 3큰술을 넣고 약불에서 2~3분 끓여 농도를 맞춥니다. 너무 되면 물을 1큰술씩 추가합니다.',
        '불을 끄고 식초를 넣어 산미를 ‘마지막에’ 정리합니다(향이 살아남). 필요하면 소금으로 간을 미세 조정합니다.',
        '들기름 요거트: 요거트에 들기름 1큰술과 소금 한 꼬집을 넣고 부드럽게 섞습니다.',
        '플레이팅: 구운 단호박 위에 들기름 요거트를 한 스푼 얹고, 생강-된장 캐러멜 소스를 줄무늬로 뿌립니다.',
        '마무리로 검은깨/잣을 뿌리고, 실험적으로는 김가루를 아주 소량 올려 감칠맛을 추가합니다(의외로 잘 
맞습니다).'
    ],
    tip='실패하는 5가지 시나리오\n1) 생강이 탄맛/쓴맛이 남음: 생강을 센 불에서 오래 볶으면 씁니다. 버터 녹인 뒤 
‘약불 30~60초’만, 색이 변하기 전에 다음 단계로.\n2) 된장 향이 텁텁하고 짜게만 느낌: 된장 양이 많거나 볶는 시간이 
짧으면 텁텁합니다. 20~30초 정도는 꼭 볶아 구수함을 내고, 짠맛은 ‘물로 농도 조절’하세요.\n3) 캐러멜이 뭉치거나 
분리됨: 조청을 넣은 뒤 물을 한 번에 많이 붓거나 불이 너무 세면 분리됩니다. 약불 유지, 물은 1큰술씩 나눠 넣고 
저으세요.\n4) 산미가 튀어 어색함: 식초를 끓이는 동안 넣으면 날카로워집니다. 반드시 불 끈 뒤 넣고, 1작은술부터 
시작해 조절.\n5) 전체 맛이 ‘달기만’ 하거나 ‘향만 강함’: 단맛이 과하면 간장/식초를 1/2작은술씩 늘려 밸런스. 생강 
향이 과하면 요거트를 늘리거나 견과(잣/호두)로 지방감을 보강하세요.'
)

해당 출력은 Pydantic 클래스 형식으로 생성됩니다.   
with_structured_output 기능을 지원하지 않는 경우, PydanticOutputParser를 사용해야 합니다.

In [18]:
from langchain_core.output_parsers import PydanticOutputParser

pydantic_parser = PydanticOutputParser(pydantic_object = Recipe)

recipe_template4 =ChatPromptTemplate([
    ('system','당신은 전세계의 이색적인 퓨전 조리법의 전문가입니다.'),
    ('user','''저는 {ingredient}를 이용한 실험적인 음식을 만들고 싶습니다. 추천해주세요!
    레시피에 대한 정보를 JSON 형식으로 출력해주세요. 결과는 한국어로 작성하세요.
     {instruction}''')
])

structured_llm2 = recipe_template4.partial(instruction = pydantic_parser.get_format_instructions()) | gpt_llm | pydantic_parser

response = structured_llm2.invoke('커피')
response

Recipe(name='에스프레소 미소 카라멜 글레이즈 연어 타코 (커피-된장 퓨전)', difficulty='중상 (불 조절과 글레이즈 농도 조절이 핵심)', origin='퓨전: 일본(된장) × 멕시코(타코) × 이탈리아(에스프레소)', ingredients=['연어 필렛 300g (껍질 있으면 더 좋음)', '에스프레소 30ml (진하게)', '된장(미소) 1.5큰술', '간장 1큰술', '흑설탕 또는 황설탕 1큰술', '라임즙 1큰술 + 라임 웨지(서빙용)', '식초(쌀식초/사과식초) 1작은술', '고추가루 또는 칠리파우더 1작은술', '마늘 1쪽(다진 것) 또는 마늘가루 1/2작은술', '생강 약간(선택)', '식용유 1큰술', '옥수수 또르띠야 6장', '양배추 채 1컵', '오이 또는 무 얇게 슬라이스 1/2컵', '고수(선택) 한 줌', '마요네즈 3큰술', '플레인 요거트 2큰술(또는 사워크림)', '추가 에스프레소 또는 진한 커피 1작은술(소스용, 선택)', '소금/후추 약간', '통깨 또는 땅콩 부순 것(토핑, 선택)'], instructions=['글레이즈 만들기: 작은 냄비에 에스프레소 30ml, 된장 1.5큰술, 간장 1큰술, 흑설탕 1큰술, 식초 1작은술, 칠리파우더 1작은술, 마늘(및 생강)을 넣고 약불에서 저어가며 3~5분 끓인다.', '농도 맞추기: 소스가 숟가락 뒷면에 얇게 코팅되는 시럽 정도가 되면 불을 끈다(너무 되직하면 물 1~2큰술로 조절).', '연어 준비: 연어에 소금/후추를 가볍게 하고, 글레이즈의 1/3을 발라 10분만 재운다(너무 오래 재우면 짜고 질감이 무를 수 있음).', '팬 시어링: 중강불로 달군 팬에 기름을 두르고 연어를 껍질면(또는 겉면)부터 3~4분 굽는다. 뒤집어 2분 굽고, 남은 글레이즈의 절반을 팬에 넣어 연어에 끼얹으며 30~60초 더 캐러멜라이즈한다(타기 직전까지).', '휴지 후 결대로 찢기: 연어를 접시에 옮겨 2분 휴지하고, 포크로 한입 크기로 결대로 가볍게 찢는다.', '커피 라임 

## [실습] LLM으로 보고서 개요 생성 후 섹션별 글 작성하기

Structured Output 구조를 활용해, LLM이 주제에 대한 구획을 먼저 구성하고    
해당 구획을 반복문이나 Batch로 각각 입력하여 긴 글을 쓰도록 만들어 보세요.

1. with_structured_output을 통해 주제에 대한 구획 작성하는 체인 `outliner` 만들기
2. 섹션별 글 작성 체인 `writer` 만들기
3. 반복문이나 batch()를 통해 `outliner`의 결과물을 `writer`에 전달하기
4. 최종 결과물 합치기

In [19]:
class Sections(BaseModel):
    topic: str = Field(description="글쓰기 주제")
    sections: list[str] = Field(description="주제에 대한 세부 섹션 개요 리스트 (최대 5개 섹션)")

In [20]:
outliner = gpt_llm.with_structured_output(Sections)
outline = outliner.invoke("""
멀티모달 LLM의 발전 과정에 대한 보고서 개요와 목차를 써줘.
각각의 개요는 병렬적 작성이 가능하도록 독립적인 내용을 담아야 하고.
개요만 보고도 내용이 구체적으로 드러나야 해.
목차는 리스트로 작성해.
""")
outline

Sections(topic='멀티모달 LLM의 발전 과정', sections=['서론: 멀티모달 LLM의 정의·범위·평가축 정립(텍스트-이미지-오디오-비디오, 입력/출력, 에이전트성)과 ‘발전’의 기준(성능, 데이터, 아키텍처, 학습전략, 정렬, 제품화)을 명확히 규정한다.', '초기 토대(2010s~2020): 멀티모달 표현학습의 기원(이미지 캡셔닝, VQA, 공동 임베딩)과 트랜스포머/대규모 사전학습의 도입이 이후 LLM-비전 결합의 전제(대규모 웹 데이터, contrastive 학습, self-supervised)를 어떻게 마련했는지 정리한다.', '비전-언어 프리트레인(2020~2022): CLIP/ALIGN류의 대규모 대조학습과 ViLT/LXMERT류의 cross-attention 방식이 ‘텍스트-이미지 정렬’을 표준화했고, 다운스트림 전이·제로샷 인식·프롬프트 기반 활용으로 이어진 과정을 비교(장단점, 계산비용, 데이터 의존성)한다.', 'LLM 중심 멀티모달 전환(2022~2023): LLM을 ‘언어 추론 코어’로 두고 비전 인코더+프로젝터를 접합하는 아키텍처(LLaVA, Flamingo, BLIP-2 등)가 등장한 배경, 인스트럭션 튜닝/대화 데이터 합성, RLHF·DPO류 정렬이 멀티모달 대화 성능을 끌어올린 메커니즘을 정리한다.', '확장과 현재 과제(2024~현재): 텍스트+이미지에서 오디오·비디오·3D·문서로 확장, 도구사용/에이전트화(검색·코드·OCR·분석), 장문 컨텍스트·메모리·스트리밍, 신뢰성(환각·근거제시)·안전(저작권/프라이버시)·평가(벤치마크 한계, 실제업무) 및 향후 로드맵(모달 통합 학습, 데이터 거버넌스, 효율화)을 제시한다.'])

In [21]:
outline.sections

['서론: 멀티모달 LLM의 정의·범위·평가축 정립(텍스트-이미지-오디오-비디오, 입력/출력, 에이전트성)과 ‘발전’의 기준(성능, 데이터, 아키텍처, 학습전략, 정렬, 제품화)을 명확히 규정한다.',
 '초기 토대(2010s~2020): 멀티모달 표현학습의 기원(이미지 캡셔닝, VQA, 공동 임베딩)과 트랜스포머/대규모 사전학습의 도입이 이후 LLM-비전 결합의 전제(대규모 웹 데이터, contrastive 학습, self-supervised)를 어떻게 마련했는지 정리한다.',
 '비전-언어 프리트레인(2020~2022): CLIP/ALIGN류의 대규모 대조학습과 ViLT/LXMERT류의 cross-attention 방식이 ‘텍스트-이미지 정렬’을 표준화했고, 다운스트림 전이·제로샷 인식·프롬프트 기반 활용으로 이어진 과정을 비교(장단점, 계산비용, 데이터 의존성)한다.',
 'LLM 중심 멀티모달 전환(2022~2023): LLM을 ‘언어 추론 코어’로 두고 비전 인코더+프로젝터를 접합하는 아키텍처(LLaVA, Flamingo, BLIP-2 등)가 등장한 배경, 인스트럭션 튜닝/대화 데이터 합성, RLHF·DPO류 정렬이 멀티모달 대화 성능을 끌어올린 메커니즘을 정리한다.',
 '확장과 현재 과제(2024~현재): 텍스트+이미지에서 오디오·비디오·3D·문서로 확장, 도구사용/에이전트화(검색·코드·OCR·분석), 장문 컨텍스트·메모리·스트리밍, 신뢰성(환각·근거제시)·안전(저작권/프라이버시)·평가(벤치마크 한계, 실제업무) 및 향후 로드맵(모달 통합 학습, 데이터 거버넌스, 효율화)을 제시한다.']

In [22]:
writer_prompt = ChatPromptTemplate([
    ('human','''보고서 주제에 대해, 하나의 섹션에 대한 전문적인 글을 작성하세요.

제목은 ##, 소목차는 ###으로 쓰고, 이외의 목차 형식은 넣지 마세요.
챕터명에 숫자를 넣지 마세요.
내용은 '입니다' 와 같은 말투로 작성하세요.

---
보고서 전체 주제: {topic}
세부 섹션 주제: {section}
''')
])
writer = writer_prompt | gpt_llm | StrOutputParser()
writer.invoke({'topic':outline.topic, 'section':outline.sections[0]})

'## 서론: 멀티모달 LLM의 정의·범위·평가축 정립과 ‘발전’의 기준\n\n멀티모달 LLM은 대규모 언어 모델을 기반으로 텍스트뿐 아니라 이미지, 오디오, 비디오 등 복수의 데이터 양식을 입력으로 수용하고, 언어적 추론과 생성 능력을 중심으로 다양한 양식의 이해·변환·생성을 수행하는 모델 계열입니다. 본 보고서는 멀티모달 LLM의 “발전 과정”을 다루기 위해, 먼저 멀티모달 LLM의 정의와 범위를 일관되게 고정하고, 비교 가능한 평가축을 정립하며, 무엇을 “발전”으로 간주할지에 대한 기준을 명확히 규정하는 것을 목표로 합니다.\n\n### 멀티모달 LLM의 정의와 범위\n\n멀티모달 LLM의 핵심은 언어 모델이 중심 오케스트레이터 역할을 수행한다는 점입니다. 즉, 입력 양식이 다양하더라도 최종적인 의미 표현, 추론, 계획, 응답 생성의 중심이 언어 공간 또는 언어 중심 잠재공간에 의해 조직되는 체계입니다. 범위 설정을 위해 본 보고서는 멀티모달 LLM을 다음의 조건을 만족하는 시스템으로 정의합니다.\n\n첫째, 입력 양식이 텍스트 단일이 아니라 최소 한 가지 이상의 비텍스트 양식(이미지, 오디오, 비디오)을 포함할 수 있어야 합니다. 둘째, 출력은 텍스트에 한정될 수 있으나, 모델 또는 시스템 차원에서 이미지·오디오·비디오와 같은 비텍스트 출력까지 확장 가능한 설계를 포함할 수 있습니다. 셋째, 단일 모델이 모든 양식을 종단간으로 처리하지 않더라도, 비전 인코더·오디오 인코더·디코더 등 모듈 결합을 통해 LLM이 멀티모달 신호를 의미적으로 결합하고 과업을 수행하는 구성이면 본 보고서의 대상에 포함합니다.\n\n이때 “멀티모달”은 단순히 입력을 여러 종류 받는 인터페이스 수준의 확장이 아니라, 서로 다른 양식 간 정보 결합(예: 이미지의 공간 정보와 텍스트 지시의 결합, 비디오의 시간 정보와 질문의 결합)이 과업 성능을 실질적으로 좌우하는 수준의 통합을 의미합니다.\n\n### 모달리티 축의 정립: 텍스트-이미지-오디오-비디오\n\n본 보고서는 멀티모달 LL

In [23]:
writer = writer_prompt.partial(topic=outline.topic) | gpt_llm | StrOutputParser()
# topic을 미리 채워 매개변수 1개
result = writer.batch(outline.sections)
result

['## 서론: 멀티모달 LLM의 정의·범위·평가축과 발전 기준\n\n멀티모달 LLM은 대규모 언어 모델을 중심으로 텍스트 이외의 감각 양식(modality) 정보를 함께 이해·생성할 수 있도록 확장된 모델 계열을 의미합니다. 전통적 LLM이 텍스트 토큰 시퀀스만을 입력·출력으로 다루는 반면, 멀티모달 LLM은 이미지, 오디오, 비디오 등 비텍스트 신호를 표현 공간으로 변환하여 언어 추론과 결합하거나, 반대로 언어적 계획을 바탕으로 비텍스트 출력을 생성하는 체계를 포함합니다. 본 보고서는 멀티모달 LLM의 발전 과정을 논의하기 위해 먼저 정의와 범위를 엄밀히 정리하고, 이후 장들에서 비교 가능한 평가축과 ‘발전’의 기준을 명시적으로 규정하는 것을 목표로 합니다.\n\n### 멀티모달 LLM의 정의\n\n멀티모달 LLM은 다음 조건을 충족하는 모델로 정의합니다. 첫째, 핵심 추론 및 지식 표현이 언어 모델(Transformer 계열 LLM)에서 수행되며, 둘째, 최소 하나 이상의 비텍스트 모달리티가 입력 또는 출력 파이프라인에 통합되어 있으며, 셋째, 모달 간 상호참조(예: 이미지 내 객체를 지칭하며 추론, 오디오 내용과 문맥의 결합, 비디오의 시간적 단서 활용)가 과제 성능에 실질적으로 기여하는 구조입니다. 이 정의는 단순한 “이미지 캡셔닝 전용 모델”이나 “음성 인식 단일 모델”처럼 LLM 중심의 범용 추론이 핵심이 아닌 시스템을 배제하고, LLM이 멀티모달 정보를 활용해 범용 과제 수행 능력을 확장하는 계열을 연구 대상으로 삼는다는 점을 분명히 합니다.\n\n### 범위 설정: 텍스트·이미지·오디오·비디오와 입력/출력 조합\n\n본 보고서에서 다루는 멀티모달 범위는 텍스트-이미지-오디오-비디오의 네 축을 기본으로 하며, 각 모달리티는 입력과 출력의 조합으로 구분하여 분석합니다.\n\n- 텍스트-이미지의 경우, 이미지 입력 기반 질의응답, 문서·차트·UI 이해, 시각적 추론(VQA, chart QA), 이미지 생성 및 편집(텍스트→이미지, 이미지→이미지)까지

In [24]:
draft = '\n\n'.join(result)
with open('result.md', 'w', encoding='utf-8') as f:
    f.write(draft)

print(draft[0:100])

## 서론: 멀티모달 LLM의 정의·범위·평가축과 발전 기준

멀티모달 LLM은 대규모 언어 모델을 중심으로 텍스트 이외의 감각 양식(modality) 정보를 함께 이해·생성할 수


<br><br>
## Runnables

LangChain 체인의 기본 구조는 `RunnableSequence` 클래스로 구성됩니다.   

이 때, 시퀀스를 구성한 llm, prompt, chain 각 모듈은 Runnables에 해당합니다.   
Runnables은 자유롭게 체인에 포함되어 결과를 연결할 수 있습니다.



이번에는, 데이터 흐름을 제어하는 특별한 Runnable인   
RunnablePassthrough와 RunnableParallel을 이용해 체인을 구성해 보겠습니다.


<br><br>
### RunnablePassthrough
RunnablePassthrough는 체인의 직전 출력을 그대로 가져옵니다.

In [25]:
from langchain_core.runnables import RunnablePassthrough

prompt1 = ChatPromptTemplate(["{director}의 대표 작품은 무엇입니까? 하나의 작품만 선택하고, 해당 작품에 대해 20자 이내로 설명하세요."])
chain1 = (
    prompt1
    | gpt_llm
    | StrOutputParser()
    | {'answer': RunnablePassthrough()})

response = chain1.invoke("봉준호")
response

{'answer': '**기생충**: 계급 갈등을 그린 블랙코미디'}

<br><br>
### RunnableParallel

RunnableParallel은 서로 다른 체인을 병렬적으로 실행하여 dict 구조로 전달합니다.

In [26]:
from langchain_core.runnables import RunnableParallel

prompt1 = ChatPromptTemplate(["색깔을 하나 알려주세요, 색깔만 출력하세요."])
prompt2 = ChatPromptTemplate(["음식을 하나 알려주세요, 음식만 출력하세요."])

chain1 = prompt1 | gpt_llm | StrOutputParser()
chain2 = prompt2 | gpt_llm | StrOutputParser()

chain3 = RunnableParallel(color = chain1, food = chain2)
# 개별 체인을 병렬 실행한 뒤, dict로 반환

chain3.invoke({})

{'color': '파란색', 'food': '비빔밥'}

## Assign()

RunnableParallel을 사용하면 중간 체인의 결과를 전달하여, 다음 체인의 결과를 함께 얻을 수 있습니다.   

In [27]:
prompt1 = ChatPromptTemplate(["잭슨빌은 어느 나라의 도시입니까?"])
prompt2 = ChatPromptTemplate(
    ["{country}의 대표적인 인물 3명을 나열하세요. 인물의 이름만 출력하세요."]
)

chain1 = prompt1 | gpt_llm | StrOutputParser()
chain2 = prompt2 | gpt_llm | StrOutputParser()

chain3 = RunnableParallel(country = chain1).assign(people = chain2)
#        {    'country': '미국'           }

chain3.invoke({})

{'country': '잭슨빌(Jacksonville)은 **미국(United States)**의 도시입니다. 보통 **미국 플로리다주(Florida)**에 있는 *잭슨빌*을 가리킵니다.',
 'people': 'Fred Durst  \nBob Hayes  \nPat Boone'}

<br><br><br><br><br><br><br><br>
chain2에서 새로운 매개변수가 추가되는 경우는 어떻게 해야 할까요?

In [28]:
prompt1 = ChatPromptTemplate(["{city}는 어느 나라의 도시인가요? 나라 이름만 출력하세요."])
prompt2 = ChatPromptTemplate(["{country}의 유명한 인물은 누가 있나요? {num} 명의 이름을 나열하세요. 사람 이름만 ,로 구분하여 나열하세요."])

chain1 = prompt1 | gpt_llm | StrOutputParser()

chain2 = (
    RunnablePassthrough.assign(country = chain1)
    # 입력받은 city, num에 country를 추가하여 전달

    | prompt2
    # country, num을 받아 실행
    | gpt_llm
    | StrOutputParser()
)

print(chain2.invoke({"city": "잭슨빌", "num": "3"}))

조지 워싱턴,마틴 루터 킹 주니어,오프라 윈프리


<br><br>
assign을 여러 개 연결할 수 있습니다.

In [29]:
chain4 = (prompt2
    | gpt_llm
    | StrOutputParser())

chain3 = RunnablePassthrough.assign(country = chain1).assign(res = chain4)

chain3.invoke({"city": "부에노스 아이레스", "num": "3"})

{'city': '부에노스 아이레스',
 'num': '3',
 'country': '아르헨티나',
 'res': '리오넬 메시, 에바 페론, 디에고 마라도나'}

<br><br><br>JsonOutputParser를 쓴다면 아래와 같이 만들 수도 있습니다.

In [30]:
prompt1 = ChatPromptTemplate(
    ["영화 배우 한명과 대표작 하나를 출력하세요. json 형식으로 출력하고, 각 항목은 actor, movie로 표시하세요."])
prompt2 = ChatPromptTemplate(["{actor}는 {movie}에서 어떤 역할을 했습니까?"])

chain1 = prompt1 | gpt_llm | JsonOutputParser()
chain2 =(
     chain1 | prompt2 | gpt_llm | StrOutputParser()
)
chain2.invoke({})

'송강호는 영화 **〈기생충〉(2019)**에서 **김기택** 역을 맡았습니다. 김기택은 **기우(최우식)**와 **기정(박소담)**의 아버지이자, 이야기가 진행되면서 **박사장(이선균)** 집의 **운전기사**로 일하게 되는 인물입니다.'

In [31]:
chain3 = prompt2 | gpt_llm | StrOutputParser()

chain4 = chain1.assign(result = chain3)

chain4.invoke({})

{'actor': '송강호',
 'movie': '기생충',
 'result': '송강호는 영화 **〈기생충〉(2019)**에서 **기택(김기택)** 역을 맡았습니다.  \n가난한 반지하에 사는 **기우(최우식), 기정(박소담)의 아버지**이자 **기택 가족의 가장**으로, 박사장(이선균) 집에 **운전기사로 취업**하면서 가족이 하나둘 그 집에 들어가게 되는 중심 인물입니다.'}